# RadarVLA：Grounding / SFT 逐帧可视化诊断

本 notebook 默认只读取 validation split。它同时展示最新一帧 Power、**未经 `doppler_valid` 遮罩的现有 unfolded Doppler 数值**、GT 道路与 Agent、Grounding Agent、SFT 微调后的 Agent，以及 SFT 自由生成的道路和轨迹。

现有权重使用旧监督口径训练：Agent 正例只到 50 m，但 50–118 m 未标注候选曾被作为背景。图中会明确画出这一边界。第一次运行 SFT 推理需要加载本地 Qwen 权重，可能耗时较长。

In [ ]:
from pathlib import Path
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('请从 RadarVLA 仓库内打开本 notebook')

from radar_vla.visual_diagnosis import DiagnosisSession

MANIFEST = ROOT / 'analysis/rebalanced_50m_domain/manifest.jsonl'
GROUNDING = ROOT / 'analysis/retrained_grounding_v2/best.pt'
SFT = ROOT / 'analysis/retrained_sft_v2/best.pt'
for path in (MANIFEST, GROUNDING, SFT):
    if not path.is_file():
        raise FileNotFoundError(path)

session = DiagnosisSession(MANIFEST, GROUNDING, SFT, split='val')
print(f'Validation samples: {len(session.index)} | device: {session.device}')
display(session.build_widget())

## 读图方法

- 绿色：GT Agent；白色/深绿：GT 道路。
- 橙色：Grounding 阶段 Agent；紫色：经过 SFT 后的 Grounding Agent。
- 青色虚线：SFT 自由生成道路；蓝色：SFT ego 轨迹。
- 虚线扇区：0–50 m、±53° 的 Agent 标注域。扇区外雷达仍是模型输入，但没有完整 Agent 负例监督。
- SFT 输出 SHORT 时，语法本身不包含道路；面板会明确显示，不应解释为地图漏检。